## ***Initials***

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json
from tqdm import tqdm
import numpy as np
import ollama

## ***Help Functions***

In [2]:
# === Embed function ===
def get_embedding(text, tokenizer, model, device):
    prompt = "Represent this sentence for retrieval: " + text
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        output = model(**inputs)
        embedding = output.last_hidden_state[:, 0]
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy()[0]

## ***Load the Data***

### ***Neighborhood Descriptions***

In [3]:
neighborhood_descriptions_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Neighborhood Descriptions\\neighborhood_descriptions_20.csv")
print(neighborhood_descriptions_df.shape)
print(neighborhood_descriptions_df.columns)
neighborhood_descriptions_df.head()

(48, 2)
Index(['Neighborhood', 'Description'], dtype='object')


,Neighborhood,Description
0,Dimini,Dimini – a spacious neighborhood (37.34 km²) w...
1,Xrisi Akti Panagias,Xrisi Akti Panagias – a low-density location (...
2,Sesklo,Sesklo – a low-density location (27.37 km²) th...
3,Agioi Anargiroi,Agioi Anargiroi – a cozy location (0.77 km²) t...
4,Aivaliotika,Aivaliotika – a neighborhood (4.85 km²) that o...


## ***Load the Embedder Model***

In [4]:
# Load the Embedder and its tokenizer
classifier_model_name = "BAAI/bge-large-en-v1.5"

In [5]:
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_model_name)
classifier_model = AutoModel.from_pretrained(classifier_model_name)

In [6]:
# Set device (CPU or CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

## ***Embed all the Neighborhood Descriptions***

In [7]:
# === Embed and collect results ===
neighborhood_embeddings = []

for _, row in neighborhood_descriptions_df.iterrows():
    neighborhood = row["Neighborhood"]
    description = row["Description"]
    
    embedding = get_embedding(description, classifier_tokenizer, classifier_model, device)

    neighborhood_embeddings.append({
        "neighborhood": neighborhood,
        "embedding": embedding.tolist()
    })

## ***Save the Data***

In [9]:
# Save to JSON
with open("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Neighborhood Description Embeddings\\neighborhoods_descriptions_embeddings_20.json", "w", encoding="utf-8") as f:
    json.dump(neighborhood_embeddings, f, ensure_ascii=False)